# 05. Evaluation: Diebold-Mariano, Mincer-Zarnowitz, regimes and calibration

Milestone 5 scores every stored model on the dev period and asks four questions of the scores:

```
which is more accurate?      Diebold-Mariano on d_t = L_a,t - L_b,t   (squared error, QLIKE)
                             long-run variance: fixed-b Bartlett, b = 0.1 (primary, measured)
                             Holm across the pairs of one (target, horizon, loss)
is it the conditional mean?  Mincer-Zarnowitz  y = alpha + beta*yhat, Wald alpha = 0, beta = 1
                             Hansen-Hodrick covariance at lag h - 1 (measured)
where does it fail?          losses by regime at the origin (VIX level, term structure)
is it biased?                k_c = sum(y*yhat)/sum(yhat^2) on targets dated <= c: a separate variant
```

The full step (after the model runs, local only for real data):

```
uv run python -m ivf.evaluation report [--print-tables]
uv run python -m ivf.evaluation --synthetic report --print-tables
```

This notebook runs three fast baselines on the last `IVF_NOTEBOOK_FOLDS` dev folds (default 8) of the
**synthetic** market and evaluates them with the same functions. **The holdout is not read**: the one
look is at the end of milestone 6 (ADR 0001 4.6, policy of 2026-10-29).

*This file is generated by `tools/build_notebooks.py`. Edit the spec there.*

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd

from ivf import protocol
from ivf.data import calendar, synthetic
from ivf.evaluation import calibration, dm, mz, regimes, report, walkforward
from ivf.features import build
from ivf.models import baselines, registry

warnings.filterwarnings("ignore", category=FutureWarning)

PROTO = protocol.load_protocol(protocol.protocol_path(synthetic=True))
IV, PRICE = synthetic.synthetic_market(PROTO)
CAL = calendar.master_calendar(PROTO)
CALENDAR_DATES = list(CAL["date"])
DEV = [f for f in walkforward.fold_schedule(CALENDAR_DATES, PROTO.dev_start, PROTO.holdout_start, PROTO.fold_len, PROTO.data_end) if f.period == "dev"]
DATES, _, IV_PRE, PRICE_PRE = walkforward.pre_holdout_inputs(CAL, IV, PRICE, PROTO.holdout_start)
N_FOLDS = int(os.environ.get("IVF_NOTEBOOK_FOLDS", "8"))
_, SETTINGS = registry.load_models()
MODELS = baselines.build_models(SETTINGS, ["rw", "hist_mean", "har_log"])
VIX = build.align(DATES, IV_PRE, "symbol", "VIX")
RUN = walkforward.run_symbol("VIX", "vix_level", VIX, DATES, CALENDAR_DATES, DEV[-N_FOLDS:], MODELS, PROTO.horizons,
                             PROTO.holdout_start, SETTINGS.min_train_sessions)
TARGETS = build.build_target_values(DATES, IV_PRE, PRICE_PRE)
LABELS = build.build_regime_labels(DATES, IV_PRE, tuple(PROTO.vix_regime_edges))
FC = RUN.forecasts.drop(columns=["y"])
print("data: SYNTHETIC | folds", N_FOLDS, "| forecast rows", len(FC), "| last readable session", DATES[-1], "< holdout_start", PROTO.holdout_start)

## 1. One Diebold-Mariano statistic by hand

`d_t` is the squared-error differential of `har_log` against `rw` at h = 21. The rectangular estimate
sums the first h - 1 autocovariances with weight 1; HLN's factor corrects its small-sample bias.

In [ ]:
# SYNTHETIC ILLUSTRATION ONLY: a 63-pair warm-up so that a few folds show a calibrated variant at all. The
# pre-registered rule, used by the report step, waits for calibration.MIN_PAIRS = 252 pairs (amended M5, review A).
ILLUSTRATION_MIN_PAIRS = 63
print(f"calibration warm-up here: {ILLUSTRATION_MIN_PAIRS} pairs (illustration); the report uses {calibration.MIN_PAIRS}")
TABLES = report.evaluate(FC, TARGETS, LABELS, PROTO.holdout_start, "dev", PROTO.document["min_cell_n"], min_pairs=ILLUSTRATION_MIN_PAIRS)
S = TABLES["_scored"]
h = 21
g = S[S.horizon == h].sort_values("origin_date")
la = dm.loss_series(g[g.model_id == "har_log"].y, g[g.model_id == "har_log"].yhat, "sq_error")
lb = dm.loss_series(g[g.model_id == "rw"].y, g[g.model_id == "rw"].yhat, "sq_error")
d = la - lb
n = d.size
x = d - d.mean()
gamma = [x[j:] @ x[: n - j] / n for j in range(h)]
lrv = gamma[0] + 2 * sum(gamma[1:])
by_hand = d.mean() / np.sqrt(lrv / n) * np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
print(f"n {n}, mean d {d.mean():.4f}, rectangular LRV {lrv:.2f}, HLN statistic by hand {by_hand:.6f}")
for method in dm.METHODS:
    r = dm.dm_test(d, h, method)
    print(f"{method:12} stat {r.dm_stat:8.4f}  p {r.p_value:.4f}  lags {r.max_lag:4d}  ({r.lrv_method})")

## 2. Does the test have the right size?

Under a null with `E[d] = 0` a 5% test should reject on about 5% of samples. Two designs at h = 21:
equally good forecasts with overlapping errors, and the state-dependent null, where one model's
accuracy follows a persistent state. (The walkthrough runs 2,000 samples per cell.)

In [ ]:
rng = np.random.default_rng(PROTO.document["seed"])
for design in ("sq_error", "state"):
    rates = dm.rejection_rates(dm.simulate_differentials(design, 2000, 21, 300, rng), 21)
    print(design, {m: round(r, 3) for m, r in rates.items()})

## 3. The tables: DM with Holm, Mincer-Zarnowitz, regimes, calibration

Sign: `d = L_a - L_b` with `model_a` first alphabetically, so a **positive** `mean_d` means `model_a` is
worse; `lower_loss` names the better one. The calibration rows here use a 63-pair warm-up, a synthetic
illustration; the pre-registered rule waits for 252 pairs.

In [ ]:
D = TABLES["dm_test"]
print(D[D.primary & (D.family == "pairs")][["horizon", "loss", "model_a", "model_b", "mean_d", "lower_loss", "dm_stat", "p_value", "p_holm"]].round(4).to_string(index=False))
Z = TABLES["mz_regression"]
print(Z[Z.primary & (Z.forecast == "mean")][["horizon", "model_id", "alpha", "beta", "wald_p", "cov_method"]].round(4).to_string(index=False))
M = TABLES["metric"]
print(M[(M.forecast == "mean") & (M.horizon == 21)][["model_id", "scheme", "regime", "n", "rmse", "qlike"]].round(4).to_string(index=False))
print(TABLES["calibration_summary"].round(4).to_string(index=False))

## 4. The holdout path, on synthetic rows only

`scored_rows` keeps a dev row only when its origin **and** its target are before `holdout_start`;
a holdout row has its origin on or after it. The real report refuses `--period holdout`.

In [ ]:
FRAME = report.attach_targets(FC, TARGETS)
dev_rows = report.scored_rows(FRAME, PROTO.holdout_start, "dev")
print("dev rows", len(dev_rows), "| latest target", dev_rows.target_date.max(), "| holdout rows", len(report.scored_rows(FRAME, PROTO.holdout_start, "holdout")))
print("holdout_looks", PROTO.document["holdout_looks"])